## Importing data:

In [6]:
import pandas as pd

train_path = '/kaggle/input/competitions/nlp-getting-started/train.csv'
test_path = '/kaggle/input/competitions/nlp-getting-started/test.csv'

train_df = pd.read_csv(train_path)
print('Training set: ',train_df.shape)
display(train_df.head(3))

test_df = pd.read_csv(test_path)
print('Test set: ',test_df.shape)
display(test_df.head(3))

Training set:  (7613, 5)


,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1


Test set:  (3263, 4)


,id,keyword,location,text
0,0,NaN,NaN,Just happened a terrible car crash
1,2,NaN,NaN,"Heard about #earthquake is different cities, s..."
2,3,NaN,NaN,"there is a forest fire at spot pond, geese are..."


In [7]:
print(f"Total samples in the training set are: {len(train_df)}")

Total samples in the training set are: 7613


In [8]:
target = 'target'

#we'll be using bag-of-word model so we will only require text column and the target
X = train_df['text'].values
y = train_df[target].values

## Preparing the data:
We'll split the train set into train/val set using **train_test_split()**, we'll have about 20% of training data in validation split.

In [9]:
from sklearn.model_selection import train_test_split

X_train,X_val,y_train,y_val = train_test_split(X,y, 
                                               test_size=0.2,
                                               stratify=y, 
                                               random_state=42)

### Creating tf.dataset objects:

In [10]:
import tensorflow as tf

train_ds = tf.data.Dataset.from_tensor_slices((X_train,y_train))
train_ds = train_ds.batch(16)

val_ds = tf.data.Dataset.from_tensor_slices((X_val,y_val))
val_ds = val_ds.batch(16)

2026-03-16 16:30:42.723286: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


### Preparing int sequences for model:

In [11]:
from tensorflow.keras.layers import TextVectorization

max_tokens = 10000

vectorizer = TextVectorization(
    max_tokens = max_tokens,
    output_mode = 'int'
)

vectorizer.adapt(X_train)

In [12]:
train_ds_int = train_ds.map(lambda x, y: (vectorizer(x),y),
                            num_parallel_calls=4)

val_ds_int = val_ds.map(lambda x, y: (vectorizer(x),y), 
                            num_parallel_calls=4)

## Sequence-Model OneHotEncoded vectors based:
The simplest way to convert our integer sequences to vector
sequences is to one-hot encode the integers (each dimension would represent one
possible term in the vocabulary). On top of these one-hot vectors, we’ll add a simple
bidirectional LSTM.

In [19]:
from tensorflow import keras
from tensorflow.keras import layers

inputs = keras.Input(shape=(None,), dtype='int64') #our input is sequence of integers
embedded = layers.Lambda(
    lambda x: tf.one_hot(x, depth=max_tokens),
    output_shape=(None, max_tokens)
) (inputs) #encodes the integers into binary 10000-D vector
x = layers.Bidirectional(layers.LSTM(32)) (embedded)
x = layers.Dropout(0.5) (x)
outputs = layers.Dense(1, activation='sigmoid') (x) #output/classification layer

model = keras.Model(inputs, outputs)
model.compile(optimizer='rmsprop',
             loss='binary_crossentropy',
             metrics=['accuracy'])
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda_1 (Lambda)               │ (None, None, 10000)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 64)             │     2,568,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,568,513 (9.80 MB)

 Trainable params: 2,568,513 (9.80 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
callbacks = [
    keras.callbacks.ModelCheckpoint('one_hot_lstm.keras', save_best_only=True)
]

model.fit(train_ds_int,
         validation_data=val_ds_int,
         callbacks=callbacks,
         epochs=10)

Epoch 1/10
381/381 ━━━━━━━━━━━━━━━━━━━━ 34s 85ms/step - accuracy: 0.6122 - loss: 0.6534 - val_accuracy: 0.7754 - val_loss: 0.4928
Epoch 2/10
381/381 ━━━━━━━━━━━━━━━━━━━━ 31s 81ms/step - accuracy: 0.7931 - loss: 0.4644 - val_accuracy: 0.8037 - val_loss: 0.4578
Epoch 3/10
381/381 ━━━━━━━━━━━━━━━━━━━━ 31s 82ms/step - accuracy: 0.8438 - loss: 0.3843 - val_accuracy: 0.7919 - val_loss: 0.4799
Epoch 4/10
381/381 ━━━━━━━━━━━━━━━━━━━━ 31s 83ms/step - accuracy: 0.8656 - loss: 0.3333 - val_accuracy: 0.8043 - val_loss: 0.4845
Epoch 5/10
381/381 ━━━━━━━━━━━━━━━━━━━━ 31s 83ms/step - accuracy: 0.8868 - loss: 0.2978 - val_accuracy: 0.7958 - val_loss: 0.4963
Epoch 6/10
381/381 ━━━━━━━━━━━━━━━━━━━━ 30s 80ms/step - accuracy: 0.9020 - loss: 0.2783 - val_accuracy: 0.7938 - val_loss: 0.5108
Epoch 7/10
381/381 ━━━━━━━━━━━━━━━━━━━━ 42s 82ms/step - accuracy: 0.9125 - loss: 0.2485 - val_accuracy: 0.7945 - val_loss: 0.5513
Epoch 8/10
381/381 ━━━━━━━━━━━━━━━━━━━━ 33s 87ms/step - accuracy: 0.9184 - loss: 0.2352 - 

This model trains very slowly, especially compared to the light
weight model from teh bag-of-words approach. This is because our inputs are quite large. This model is bound to perform badly on test data since our valiation score never reached as high as teh bag-of-words models.

Clearly, using one-hot encoding to turn words into vectors, which was the simplest
thing we could do, wasn’t a great idea. There’s a better way: **word embeddings**. 

There are two ways
to obtain word embeddings:
* Learn word embeddings jointly with the main task you care about (such as doc
ument classification or sentiment prediction). In this setup, you start with ran
dom word vectors and then learn word vectors in the same way you learn the
weights of a neural network.
* Load into your model word embeddings that were precomputed using a differ
ent machine learning task than the one you’re trying to solve. These are called
**pretrained word embeddings**.

## Learning word Embeddings with Embedding layer:

In [23]:
inputs = keras.Input(shape=(None,), dtype="int64")
embedded = layers.Embedding(input_dim=max_tokens, output_dim=256)(inputs)
x = layers.Bidirectional(layers.LSTM(32))(embedded)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs, outputs)
model.compile(optimizer="rmsprop",
      loss="binary_crossentropy",
      metrics=["accuracy"])
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, None, 256)      │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 64)             │        73,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,634,049 (10.05 MB)

 Trainable params: 2,634,049 (10.05 MB)

 Non-trainable params: 0 (0.00 B)

In [24]:
callbacks = [
    keras.callbacks.ModelCheckpoint("embeddings_bidir_gru.keras",
   save_best_only=True)
]

model.fit(train_ds_int, 
          validation_data=val_ds_int, 
          epochs=10,
          callbacks=callbacks)

Epoch 1/10
381/381 ━━━━━━━━━━━━━━━━━━━━ 9s 19ms/step - accuracy: 0.6535 - loss: 0.6213 - val_accuracy: 0.7873 - val_loss: 0.4622
Epoch 2/10
381/381 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step - accuracy: 0.8170 - loss: 0.4276 - val_accuracy: 0.8102 - val_loss: 0.4471
Epoch 3/10
381/381 ━━━━━━━━━━━━━━━━━━━━ 7s 18ms/step - accuracy: 0.8591 - loss: 0.3583 - val_accuracy: 0.7978 - val_loss: 0.4777
Epoch 4/10
381/381 ━━━━━━━━━━━━━━━━━━━━ 7s 18ms/step - accuracy: 0.8807 - loss: 0.3093 - val_accuracy: 0.7899 - val_loss: 0.5172
Epoch 5/10
381/381 ━━━━━━━━━━━━━━━━━━━━ 10s 18ms/step - accuracy: 0.9012 - loss: 0.2686 - val_accuracy: 0.7899 - val_loss: 0.5325
Epoch 6/10
381/381 ━━━━━━━━━━━━━━━━━━━━ 7s 18ms/step - accuracy: 0.9156 - loss: 0.2356 - val_accuracy: 0.7800 - val_loss: 0.6327
Epoch 7/10
381/381 ━━━━━━━━━━━━━━━━━━━━ 7s 18ms/step - accuracy: 0.9335 - loss: 0.2036 - val_accuracy: 0.7676 - val_loss: 0.7194
Epoch 8/10
381/381 ━━━━━━━━━━━━━━━━━━━━ 7s 18ms/step - accuracy: 0.9474 - loss: 0.1703 - val_acc

Validation accuracy increased as well as the speed of processing increased in this case.

### Predicting on test data:

In [25]:
test_ds = tf.data.Dataset.from_tensor_slices((test_df['text'].values))
test_ds = test_ds.batch(32)
test_ds_int = test_ds.map(lambda x: vectorizer(x))

In [26]:
model = keras.models.load_model("embeddings_bidir_gru.keras")
y_preds = model.predict(test_ds_int)
y_preds = y_preds.flatten()
y_preds = (y_preds > 0.5).astype(int)

102/102 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step


In [27]:
submission = pd.DataFrame({
    'id': test_df['id'],
    target: y_preds
})
submission.to_csv('submission_wordEmbeddings.csv', index=False)

This approach/model achived a test score of 0.79374 heighest so far in DeepLearning experimets.

## Using pretrained word Embedding:
Sometimes you have so little training data available that you can’t use your data alone
to learn an appropriate task-specific embedding of your vocabulary. In such cases,
instead of learning word embeddings jointly with the problem you want to solve, you
can load embedding vectors from a precomputed embedding space that you know is
highly structured and exhibits useful properties—one that captures generic aspects of
language structure.

Here we'll use **Global Vectors for Word Representation (GloVe)**

First, let’s download the GloVe word embeddings precomputed on the 2014
English Wikipedia dataset. It’s an 822 MB zip file containing 100-dimensional embed
ding vectors for 400,000 words (or non-word tokens).

In [30]:
!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip glove.6B.zip

--2026-03-16 18:06:12--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2026-03-16 18:06:12--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2026-03-16 18:06:13--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glov

In [32]:
# Parsing the GloVe word-embeddings file
import numpy as np
path_to_glove_file = '/kaggle/working/glove.6B.100d.txt'

embeddings_index = {}
with open(path_to_glove_file) as f:
    for line in f:
        word, coefs = line.split(maxsplit=1)
        coefs = np.fromstring(coefs, 'f', sep=' ')
        embeddings_index[word] = coefs

print(f"Found {len(embeddings_index)} word vectors.")

Found 400000 word vectors.


In [33]:
# Preparing the GloVe word-embeddings matrix
embedding_dim = 100 

vocabulary = vectorizer.get_vocabulary() #retriving the vocabulary indexed by our previous TextVectorization layer
word_index = dict(zip(vocabulary, range(len(vocabulary))))

embedding_matrix = np.zeros((max_tokens, embedding_dim))  
for word, i in word_index.items():
    if i < max_tokens:
        embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:   
        embedding_matrix[i] = embedding_vector

In [34]:
embedding_layer = layers.Embedding(
    max_tokens,
    embedding_dim,
    embeddings_initializer=keras.initializers.Constant(embedding_matrix),
    trainable=False,
    mask_zero=True,
)

In [35]:
# Model
inputs = keras.Input(shape=(None,), dtype="int64")
embedded = embedding_layer(inputs)
x = layers.Bidirectional(layers.LSTM(32))(embedded)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs, outputs)
model.compile(optimizer="rmsprop",
      loss="binary_crossentropy",
      metrics=["accuracy"])
model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 100) │  1,000,000 │ input_layer_3[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, None)      │          0 │ input_layer_3[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_3     │ (None, 64)        │     34,048 │ embedding_1[0][0… │
│ (Bidirectional)     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 64)        │          0 │ bidirectional_3[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 1)         │         65 │ dropout_3[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,034,113 (3.94 MB)

 Trainable params: 34,113 (133.25 KB)

 Non-trainable params: 1,000,000 (3.81 MB)

In [36]:
callbacks = [
    keras.callbacks.ModelCheckpoint("glove_embeddings_sequence_model.keras",
   save_best_only=True)
]

model.fit(train_ds_int, 
          validation_data=val_ds_int, 
          epochs=10,
          callbacks=callbacks)

Epoch 1/10
381/381 ━━━━━━━━━━━━━━━━━━━━ 9s 18ms/step - accuracy: 0.7114 - loss: 0.5612 - val_accuracy: 0.7984 - val_loss: 0.4440
Epoch 2/10
381/381 ━━━━━━━━━━━━━━━━━━━━ 7s 18ms/step - accuracy: 0.7934 - loss: 0.4582 - val_accuracy: 0.8070 - val_loss: 0.4385
Epoch 3/10
381/381 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - accuracy: 0.8048 - loss: 0.4320 - val_accuracy: 0.7873 - val_loss: 0.4554
Epoch 4/10
381/381 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step - accuracy: 0.8164 - loss: 0.4155 - val_accuracy: 0.8050 - val_loss: 0.4378
Epoch 5/10
381/381 ━━━━━━━━━━━━━━━━━━━━ 7s 18ms/step - accuracy: 0.8291 - loss: 0.3983 - val_accuracy: 0.8129 - val_loss: 0.4335
Epoch 6/10
381/381 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - accuracy: 0.8377 - loss: 0.3795 - val_accuracy: 0.7905 - val_loss: 0.4818
Epoch 7/10
381/381 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - accuracy: 0.8441 - loss: 0.3683 - val_accuracy: 0.8004 - val_loss: 0.4925
Epoch 8/10
381/381 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - accuracy: 0.8560 - loss: 0.3555 - val_acc

### Predicting on test data:

In [37]:
model = keras.models.load_model("glove_embeddings_sequence_model.keras")
y_preds = model.predict(test_ds_int)
y_preds = y_preds.flatten()
y_preds = (y_preds > 0.5).astype(int)

102/102 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step


In [38]:
submission = pd.DataFrame({
    'id': test_df['id'],
    target: y_preds
})
submission.to_csv('submission_Glove_Embeddings.csv', index=False)

The model gave 0.80294 accuracy on test data which is best so far.